<a href="https://colab.research.google.com/github/ZarinRitu32/Soft_Computing_project/blob/main/Copy_of_Miopia_dataset_gen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm, truncnorm
from sklearn.neighbors import NearestNeighbors

# Set seed for exact reproducibility
SEED = 42
np.random.seed(SEED)
TARGET_N = 1500  # Desired total dataset size

# -------------------------------------------------------------------------
# STEP 1: Load or Mock Real Survey Dataset (N=73)
# -------------------------------------------------------------------------
# If you have a file named 'my_survey_73.csv', uncomment the line below:
# survey_73_df = pd.read_csv('my_survey_73.csv')

# Mocking the 73 real survey responses matching your collected features
survey_mock = {
    'age': np.random.randint(6, 22, size=73),
    'gender': np.random.choice([0, 1], size=73, p=[0.48, 0.52]),
    'parental_myopia_count': np.random.choice([0, 1, 2], size=73, p=[0.45, 0.38, 0.17]),
    'screen_hours_daily': np.round(np.random.gamma(shape=2, scale=1.8, size=73), 1),
    'continuous_reading_min': np.round(np.random.uniform(15, 120, size=73), 0),
    'outdoor_hours_daily': np.round(np.random.exponential(scale=1.2, size=73), 1),
    'diet_sugary_freq': np.random.choice([0, 1], size=73, p=[0.68, 0.32])
}
survey_73_df = pd.DataFrame(survey_mock)

# Enforce feature constraints on pilot survey
survey_73_df['screen_hours_daily'] = survey_73_df['screen_hours_daily'].clip(0.5, 12.0)
survey_73_df['outdoor_hours_daily'] = survey_73_df['outdoor_hours_daily'].clip(0.0, 6.0)

# -------------------------------------------------------------------------
# STEP 2: Expand Survey Data (N=73 -> N=1500) using Nearest Neighbor Resampling
# -------------------------------------------------------------------------
# Sample indices with replacement from the N=73 pilot survey
resampled_indices = np.random.choice(survey_73_df.index, size=TARGET_N, replace=True)
expanded_survey = survey_73_df.iloc[resampled_indices].copy().reset_index(drop=True)

# Add controlled numerical jittering to continuous survey features so rows are not identical
expanded_survey['screen_hours_daily'] = np.round(
    expanded_survey['screen_hours_daily'] + np.random.normal(0, 0.2, size=TARGET_N), 1
).clip(0.5, 12.0)

expanded_survey['continuous_reading_min'] = np.round(
    expanded_survey['continuous_reading_min'] + np.random.normal(0, 5.0, size=TARGET_N), 0
).clip(10, 180)

expanded_survey['outdoor_hours_daily'] = np.round(
    expanded_survey['outdoor_hours_daily'] + np.random.normal(0, 0.15, size=TARGET_N), 1
).clip(0.0, 6.0)

# -------------------------------------------------------------------------
# STEP 3: Synthesize Clinical Biometrics Conditioned on Survey Features
# -------------------------------------------------------------------------
# Correlations between biometrics and survey features derived from literature
# High risk factors (high screen, low outdoor, 2 myopic parents) shift Axial Length higher

risk_score = (
    0.35 * (expanded_survey['age'] / 22.0) +
    0.40 * (expanded_survey['parental_myopia_count'] / 2.0) +
    0.30 * (expanded_survey['screen_hours_daily'] / 12.0) +
    0.25 * (expanded_survey['continuous_reading_min'] / 180.0) -
    0.28 * (expanded_survey['outdoor_hours_daily'] / 6.0) +
    0.15 * expanded_survey['diet_sugary_freq']
)

# Standardize risk score to zero mean
risk_score_std = (risk_score - risk_score.mean()) / risk_score.std()

# Function to generate truncated normal biometrics with conditional shifts
def generate_bounded_biometric(mean, sd, low, high, risk_effect, noise_scale):
    latent_val = mean + (risk_effect * risk_score_std) + np.random.normal(0, noise_scale, size=TARGET_N)
    return np.clip(latent_val, low, high)

# 1. Axial Length (mm): Positively correlated with myopia risk (r ≈ 0.78)
expanded_survey['axial_length_mm'] = np.round(
    generate_bounded_biometric(mean=24.12, sd=1.15, low=20.50, high=28.50, risk_effect=0.65, noise_scale=0.70), 2
)

# 2. Corneal Power (D): Inverse relationship with Axial Length via emmetropization (r ≈ -0.42)
corneal_latent = 43.25 - 0.35 * (expanded_survey['axial_length_mm'] - 24.12) + np.random.normal(0, 1.10, size=TARGET_N)
expanded_survey['corneal_power_d'] = np.round(np.clip(corneal_latent, 39.00, 47.00), 2)

# 3. Anterior Chamber Depth (mm): Positive correlation with Axial Length (r ≈ 0.55)
acd_latent = 3.58 + 0.12 * (expanded_survey['axial_length_mm'] - 24.12) + np.random.normal(0, 0.20, size=TARGET_N)
expanded_survey['anterior_chamber_depth'] = np.round(np.clip(acd_latent, 2.60, 4.30), 2)

# 4. Spherical Equivalent (SE in D): Highly inverse to Axial Length (r ≈ -0.78) and Myopia Risk
se_latent = (
    0.50
    - 1.85 * (expanded_survey['axial_length_mm'] - 24.12)
    - 0.45 * risk_score_std
    + np.random.normal(0, 0.60, size=TARGET_N)
)
expanded_survey['spherical_equivalent_d'] = np.round(np.clip(se_latent, -10.00, 4.00), 2)

# -------------------------------------------------------------------------
# STEP 4: Create Binary Classification Target Label
# -------------------------------------------------------------------------
# Standard Clinical Definition: Myopia present if Spherical Equivalent <= -0.50 D
expanded_survey['is_myopic'] = (expanded_survey['spherical_equivalent_d'] <= -0.50).astype(int)

# -------------------------------------------------------------------------
# STEP 5: Reorder Columns and Save Full Dataset to CSV
# -------------------------------------------------------------------------
final_column_order = [
    # Demographics & Family
    'age', 'gender', 'parental_myopia_count',
    # Behaviors & Lifestyle (Survey Features)
    'screen_hours_daily', 'continuous_reading_min', 'outdoor_hours_daily', 'diet_sugary_freq',
    # Clinical Biometrics (Literature Synthesized)
    'axial_length_mm', 'corneal_power_d', 'anterior_chamber_depth',
    # Outcome Targets
    'spherical_equivalent_d', 'is_myopic'
]

full_dataset_df = expanded_survey[final_column_order]

# Save output to CSV
output_filename = "full_myopia_hybrid_dataset_N1500.csv"
full_dataset_df.to_csv(output_filename, index=False)

print(f"Full dataset generated and saved as '{output_filename}'")
print(f"Total Rows: {len(full_dataset_df)} | Total Features: {full_dataset_df.shape[1]}")
print("\nClass Balance ('is_myopic'):")
print(full_dataset_df['is_myopic'].value_counts(normalize=True))
print("\nFirst 5 Rows of the Full Dataset:")
print(full_dataset_df.head())

Full dataset generated and saved as 'full_myopia_hybrid_dataset_N1500.csv'
Total Rows: 1500 | Total Features: 12

Class Balance ('is_myopic'):
is_myopic
0    0.688
1    0.312
Name: proportion, dtype: float64

First 5 Rows of the Full Dataset:
   age  gender  parental_myopia_count  screen_hours_daily  \
0    9       0                      0                 5.7   
1   17       0                      1                 2.2   
2   15       0                      1                 0.7   
3   13       1                      0                 4.2   
4    8       1                      2                 1.6   

   continuous_reading_min  outdoor_hours_daily  diet_sugary_freq  \
0                    53.0                  1.4                 0   
1                    73.0                  0.1                 0   
2                    44.0                  0.7                 0   
3                    82.0                  2.6                 1   
4                    87.0                  0.7    